In [1]:
import torch
import pandas as pd
import numpy as np
import plotly.express as px
from transformer_time_series_encoder_only import (
    StockInformerEncoderOnly,
    train_model, build_target,
)
from components import create_dataloaders,TrainConfig,inverse_transform,init_weights
import random

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [3]:
# ==============================
# Load data and define config
# ==============================

csv_path = r"D:\Quan\Quants\Neural Network\financial_attention\1h_data_20220101_20250601.csv"
closes = pd.read_csv(csv_path, index_col=0, parse_dates=True)[['SOL', 'ETH', 'BTC','ADA','XRP','LTC','TRX','LINK','DOT','DOGE']]
closes = np.log(closes/closes.shift(1)).dropna()

config = {
    "d_input": len(closes.columns),
    "d_model": 64,
    "n_heads": 4,
    "d_ff": 256,
    "enc_layers": 3,
    "dropout": 0.05,
    "distill": False,
    "enc_len": 96,
    "pred_len": 1,  # Always 1-step ahead
    "factor": 5,
    "use_time_embedding": False,
    "norm_mode": "post",
    "attention_type": "prob",   # Options: "prob" or "full"
    "target_type": "change",    # Options: "change" or "price"
}
set_seed(42)
train_loader, val_loader, scaler, asset_idx = create_dataloaders(closes, enc_len=config["enc_len"],
                                                                    pred_len=config["pred_len"],
                                                                    batch_size=32,val_batch_size=32, val_ratio=0.1, asset_name="SOL")
print(f"✅ Data ready: {len(train_loader.dataset)} training samples, {len(val_loader.dataset)} validation samples.")
model = StockInformerEncoderOnly(config, asset_index=asset_idx)

model.apply(init_weights)

✅ Data ready: 26274 training samples, 2931 validation samples.


StockInformerEncoderOnly(
  (enc_embedding): Linear(in_features=10, out_features=64, bias=True)
  (pos_enc): PositionalEncoding()
  (encoder): Encoder(
    (layers): ModuleList(
      (0-2): 3 x EncoderLayer(
        (attn): ProbSparseSelfAttention(
          (q_proj): Linear(in_features=64, out_features=64, bias=True)
          (k_proj): Linear(in_features=64, out_features=64, bias=True)
          (v_proj): Linear(in_features=64, out_features=64, bias=True)
          (o_proj): Linear(in_features=64, out_features=64, bias=True)
          (attn_dropout): Dropout(p=0.05, inplace=False)
          (out_dropout): Dropout(p=0.05, inplace=False)
        )
        (ff): Sequential(
          (0): Linear(in_features=64, out_features=256, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=256, out_features=64, bias=True)
        )
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine

In [4]:
# training config
tcfg = TrainConfig(
    learning_rate=2e-5,
    weight_decay=0.01,
    max_steps=10000,
    warmup_steps=200,
    use_amp=True,
    device="cuda",
    patience=20,  # Stop if no improvement for validation checks
    min_delta=0.0000001  # Minimum improvement required
)
model, train_hist, val_hist, steps_hist,best_val_loss,best_val_step = train_model(model, train_loader, val_loader, tcfg, asset_index=asset_idx)

total_learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total learnable parameters: {total_learnable_params:,}")

2025-11-13 18:01:54,574 | INFO | [Step     0] train_loss=1.933285 | lr=0.000e+00 | samples/s=85.7
2025-11-13 18:01:54,780 | INFO | [Step    10] train_loss=1.054341 | lr=1.000e-06 | samples/s=156.6
2025-11-13 18:01:54,988 | INFO | [Step    20] train_loss=0.783076 | lr=2.000e-06 | samples/s=153.9
2025-11-13 18:01:55,195 | INFO | [Step    30] train_loss=0.551592 | lr=3.000e-06 | samples/s=154.7
2025-11-13 18:01:55,400 | INFO | [Step    40] train_loss=1.460785 | lr=4.000e-06 | samples/s=157.4
2025-11-13 18:01:55,627 | INFO | [Step    50] train_loss=1.035301 | lr=5.000e-06 | samples/s=141.2
2025-11-13 18:01:55,840 | INFO | [Step    60] train_loss=1.432972 | lr=6.000e-06 | samples/s=151.0
2025-11-13 18:01:56,039 | INFO | [Step    70] train_loss=1.449185 | lr=7.000e-06 | samples/s=161.5
2025-11-13 18:01:56,232 | INFO | [Step    80] train_loss=0.381532 | lr=8.000e-06 | samples/s=167.2
2025-11-13 18:01:56,431 | INFO | [Step    90] train_loss=0.469728 | lr=9.000e-06 | samples/s=160.6
2025-11-13 

Total learnable parameters: 150,721


In [5]:
px.line(steps_hist, labels={'x':'Training Steps', 'y':'Training Loss'}, title='Training Loss over Steps').show()

In [6]:
# ==============================
# Evaluate and plot results (aligned by target timestamps)
# ==============================
_, val_loader_1, scaler, _ = create_dataloaders(closes, enc_len=config["enc_len"],
                                                                    pred_len=config["pred_len"],
                                                                    batch_size=32,val_batch_size=1, val_ratio=0.1, asset_name="SOL")

device = torch.device(tcfg.device)
model = model.to(device).eval()

enc_len = config["enc_len"]
pred_len = config["pred_len"]
val_ratio_used = 0.1 # must match the value passed to create_dataloaders

# Rebuild the validation slice exactly like create_dataloaders
n = len(closes)
n_val = int(n * val_ratio_used)
val_df = closes.iloc[-(n_val + enc_len + pred_len):]

# Number of sliding-window samples produced by the val dataset
num_samples = len(val_df) - (enc_len + pred_len) + 1

# Target timestamps for each sample: i + enc_len
target_times = val_df.index[enc_len: enc_len + num_samples]

# Collect predictions/targets and also previous/target scaled values for price reconstruction
preds, targets = [], []
prev_scaled_list, tgt_scaled_list = [], []
use_amp = (tcfg.use_amp and device.type == "cuda")
with torch.inference_mode():
    for val_seqs, val_times in val_loader_1:
        val_seqs, val_times = val_seqs.to(device), val_times.to(device)
        # Previous and target scaled values for the asset column
        prev_scaled = val_seqs[:, enc_len - 1, asset_idx]
        tgt_scaled = val_seqs[:, enc_len + pred_len - 1, asset_idx]
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            y_pred = model(val_seqs, val_times) # [B, 1]
            y_true = build_target(val_seqs, asset_idx, getattr(model, "target_type", "change")) # [B, 1]
        preds.append(y_pred.float().cpu().detach().numpy())
        targets.append(y_true.float().cpu().numpy())
        prev_scaled_list.append(prev_scaled.float().cpu().numpy())
        tgt_scaled_list.append(tgt_scaled.float().cpu().numpy())

preds = np.concatenate(preds, axis=0).flatten()
targets = np.concatenate(targets, axis=0).flatten()
prev_scaled_all = np.concatenate(prev_scaled_list, axis=0).flatten()
tgt_scaled_all = np.concatenate(tgt_scaled_list, axis=0).flatten()

# Safety: lengths should match number of samples/timestamps
assert len(preds) == len(targets) == len(target_times) == len(prev_scaled_all) == len(tgt_scaled_all)

# Plotting branch

preds_plot = inverse_transform(preds, scaler, asset_idx, config["d_input"])
targets_plot = inverse_transform(targets, scaler, asset_idx, config["d_input"])
y_label, title = "Price", "Predicted vs Actual Price (Validation Set)"
# else:
#     # Target_type == 'change': reconstruct next price from scaled values and inverse-transform
#     curr_scaled_pred = prev_scaled_all * np.exp(preds)
#     preds_plot = inverse_transform(curr_scaled_pred, scaler, asset_idx, config["d_input"])
#     targets_plot = inverse_transform(tgt_scaled_all, scaler, asset_idx, config["d_input"])
#     y_label, title = "Price", "Predicted vs Actual Price (Validation Set)"

# Build timestamp-aligned plot
df_plot = pd.DataFrame({
"Time": target_times,
"Actual": targets_plot,
"Predicted": preds_plot,
})
df_plot = df_plot.melt(id_vars="Time", value_vars=["Actual", "Predicted"],
var_name="Type", value_name=y_label)

fig = px.line(df_plot, x="Time", y=y_label, color="Type", title=title)
fig.show()

In [7]:
preds

array([ 8.8882446e-03, -1.0251999e-05, -8.9492798e-03, ...,
       -1.4183044e-02,  6.7353249e-05, -7.0533752e-03],
      shape=(2931,), dtype=float32)

In [8]:
preds_plot

array([ 1.12732385e-04,  1.03328851e-05, -9.25330277e-05, ...,
       -1.52760604e-04,  1.12259280e-05, -7.07158955e-05], shape=(2931,))

In [9]:
targets_plot[:10]

array([-0.0058386 ,  0.00180351,  0.00377243, -0.00663316,  0.00903832,
        0.00939015,  0.00198842,  0.01257338,  0.00110813, -0.01324977])

In [10]:
preds_plot[:10]

array([ 1.12732385e-04,  1.03328851e-05, -9.25330277e-05, -2.20450781e-04,
        3.24765061e-05, -3.15620887e-04, -3.47051605e-04, -9.78007458e-05,
       -3.94812248e-04, -2.94023243e-04])

In [11]:
a = targets_plot
p = preds_plot
corr_aligned = np.corrcoef(p, a)[0, 1]
corr_shifted = np.corrcoef(p[1:], a[:-1])[0, 1]
print("corr aligned:", corr_aligned, "corr shifted:", corr_shifted)

corr aligned: 0.0021626051003995127 corr shifted: -0.9457319801711342
